# COCO Bridge — V6.0.5 고정 split 보존 버전

목적: YOLO native `labels/train|val/*.txt`를 COCO JSON으로 변환하되, **기존 V6.0.5 Train 9,763 / Val 2,433 split을 절대 다시 나누지 않는다.**

생성 파일:
- `labels/annotations.json` : 전체 12,196장 COCO JSON (각 image에 `split` 필드 포함)
- `labels/annotations_train.json` : 기존 Train 9,763장만
- `labels/annotations_val.json` : 기존 Val 2,433장만

COCO `category_id`는 YOLO class id(0~117)를 사용하고, `categories[].original_category_id`에 원래 category id를 함께 보존한다.

> **V3 경계 보정:** YOLO 정규화 좌표 반올림 때문에 0/960 경계를 0.01px 미만으로 살짝 넘는 bbox는 경계로 클립한다. 그보다 큰 초과는 데이터 오류로 간주해 계속 중단한다.


In [1]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
import csv, json
from pathlib import Path
from tqdm.auto import tqdm

WEEK2_ROOT = Path("/content/drive/MyDrive/baby_kangaroo/week2")
BUNDLE_DIR = WEEK2_ROOT / "데이터전처리" / "yolo_전처리" / "pill_yolo_full_v6_0_preprocessed"
PIPELINE_MANIFEST_DIR = WEEK2_ROOT / "공통파이프라인" / "manifests"

CLASS_MAPPING_CSV = BUNDLE_DIR / "class_mapping.csv"
DATASET_MANIFEST_CSV = BUNDLE_DIR / "dataset_manifest.csv"
LABELS_DIR = BUNDLE_DIR / "labels"
IMAGES_DIR = BUNDLE_DIR / "images"

EXPECTED_IMAGES = 12196
EXPECTED_OBJECTS = 46394
EXPECTED_CLASSES = 118
EXPECTED_TRAIN = 9763
EXPECTED_VAL = 2433
EXPECTED_SIZE = 960

for p in [BUNDLE_DIR, CLASS_MAPPING_CSV, DATASET_MANIFEST_CSV, LABELS_DIR, IMAGES_DIR, PIPELINE_MANIFEST_DIR]:
    print(p, '->', p.exists())
    if not p.exists():
        raise FileNotFoundError(p)


/content/drive/MyDrive/baby_kangaroo/week2/데이터전처리/yolo_전처리/pill_yolo_full_v6_0_preprocessed -> True
/content/drive/MyDrive/baby_kangaroo/week2/데이터전처리/yolo_전처리/pill_yolo_full_v6_0_preprocessed/class_mapping.csv -> True
/content/drive/MyDrive/baby_kangaroo/week2/데이터전처리/yolo_전처리/pill_yolo_full_v6_0_preprocessed/dataset_manifest.csv -> True
/content/drive/MyDrive/baby_kangaroo/week2/데이터전처리/yolo_전처리/pill_yolo_full_v6_0_preprocessed/labels -> True
/content/drive/MyDrive/baby_kangaroo/week2/데이터전처리/yolo_전처리/pill_yolo_full_v6_0_preprocessed/images -> True
/content/drive/MyDrive/baby_kangaroo/week2/공통파이프라인/manifests -> True


In [3]:
# 체크포인트 manifest에서 실제 split fingerprint를 읽고, 그 fingerprint의 preserved_split을 사용한다.
checkpoint_manifests = sorted(PIPELINE_MANIFEST_DIR.glob("yolo*_*.json"))
if not checkpoint_manifests:
    raise FileNotFoundError("No YOLO checkpoint manifests found")

split_fps = set()
for p in checkpoint_manifests:
    try:
        d = json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        continue
    if d.get("training_completed") is True and d.get("split_fingerprint"):
        split_fps.add(d["split_fingerprint"])

if len(split_fps) != 1:
    raise RuntimeError(f"Expected exactly one split fingerprint, got: {split_fps}")
SPLIT_FINGERPRINT = next(iter(split_fps))
SPLIT_PATH = PIPELINE_MANIFEST_DIR / f"preserved_split_{SPLIT_FINGERPRINT[:16]}.json"
if not SPLIT_PATH.is_file():
    raise FileNotFoundError(SPLIT_PATH)

split_contract = json.loads(SPLIT_PATH.read_text(encoding="utf-8"))
if split_contract.get("split_policy") != "preserve_exact_upstream":
    raise RuntimeError(f"Unexpected split policy: {split_contract.get('split_policy')}")
if split_contract.get("independent_test_available") is not False:
    raise RuntimeError("This pipeline should not create an independent test split")

train_files = list(split_contract["train_files"])
val_files = list(split_contract["val_files"])
if len(train_files) != EXPECTED_TRAIN or len(val_files) != EXPECTED_VAL:
    raise RuntimeError(f"Split count mismatch: train={len(train_files)}, val={len(val_files)}")
if set(train_files) & set(val_files):
    raise RuntimeError("Train/Val leakage detected")

print('split fingerprint:', SPLIT_FINGERPRINT)
print('train:', len(train_files), 'val:', len(val_files))


split fingerprint: cc5d16d3fe042c5297aaad5c8f2fe469ebfecfb1268223f9a4f3465bcc823154
train: 9763 val: 2433


In [4]:
# class mapping -> COCO categories
categories = []
with CLASS_MAPPING_CSV.open(encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        categories.append({
            "id": int(row["yolo_class_id"]),
            "original_category_id": int(row["original_category_id"]),
            "name": row.get("normalized_class_name") or row.get("class_name") or str(row["original_category_id"]),
        })
categories.sort(key=lambda x: x["id"])
if [c["id"] for c in categories] != list(range(EXPECTED_CLASSES)):
    raise RuntimeError("YOLO class ids must be exactly 0..117")
print('categories:', len(categories))


categories: 118


In [5]:
# dataset manifest를 file_name 기준으로 인덱싱하고 preserved split과 1:1 일치하는지 검증한다.
with DATASET_MANIFEST_CSV.open(encoding="utf-8-sig") as f:
    manifest_rows = list(csv.DictReader(f))
if len(manifest_rows) != EXPECTED_IMAGES:
    raise RuntimeError(f"dataset_manifest rows: {len(manifest_rows)}")
manifest_by_name = {r["file_name"]: r for r in manifest_rows}
if len(manifest_by_name) != EXPECTED_IMAGES:
    raise RuntimeError("Duplicate file_name in dataset_manifest.csv")

expected_all = set(train_files) | set(val_files)
if set(manifest_by_name) != expected_all:
    missing = sorted(expected_all - set(manifest_by_name))[:10]
    extra = sorted(set(manifest_by_name) - expected_all)[:10]
    raise RuntimeError(f"Manifest/preserved split mismatch. missing={missing}, extra={extra}")

print('dataset_manifest exactly matches preserved split.')


dataset_manifest exactly matches preserved split.


In [6]:
def convert_split(file_names, split_name, image_id_start=1, ann_id_start=1):
    images, annotations = [], []
    image_id = image_id_start
    ann_id = ann_id_start
    object_count = 0

    for file_name in tqdm(file_names, desc=f"COCO {split_name}", unit="image"):
        row = manifest_by_name[file_name]
        width = int(row.get("output_width") or EXPECTED_SIZE)
        height = int(row.get("output_height") or EXPECTED_SIZE)
        if width != EXPECTED_SIZE or height != EXPECTED_SIZE:
            raise RuntimeError(f"Unexpected size for {file_name}: {width}x{height}")

        # COCO file_name은 basename 그대로 둔다. split은 별도 필드로 보존한다.
        images.append({
            "id": image_id,
            "file_name": file_name,
            "width": width,
            "height": height,
            "split": split_name,
        })

        img_path = IMAGES_DIR / split_name / file_name
        if not img_path.is_file():
            raise FileNotFoundError(img_path)

        label_path = LABELS_DIR / split_name / f"{Path(file_name).stem}.txt"
        expected_obj = int(row.get("object_count", 0) or 0)
        if not label_path.is_file():
            if expected_obj != 0:
                raise FileNotFoundError(f"Missing label with object_count>0: {label_path}")
            image_id += 1
            continue

        actual_obj = 0
        for line_no, raw in enumerate(label_path.read_text(encoding="utf-8").splitlines(), 1):
            raw = raw.strip()
            if not raw:
                continue
            parts = raw.split()
            if len(parts) != 5:
                raise RuntimeError(f"Invalid YOLO label {label_path}:{line_no}")
            cls_f, cx, cy, bw, bh = map(float, parts)
            cls_id = int(cls_f)
            if cls_f != cls_id or not 0 <= cls_id < EXPECTED_CLASSES:
                raise RuntimeError(f"Invalid class id {cls_f} at {label_path}:{line_no}")
            if not all(0.0 <= v <= 1.0 for v in [cx, cy, bw, bh]):
                raise RuntimeError(f"Non-normalized bbox at {label_path}:{line_no}")

            # YOLO txt는 정규화 좌표를 소수점으로 저장하므로,
            # 이미지 경계에 정확히 붙은 box가 역변환 시 ±0.001px 정도 벗어날 수 있다.
            # 이런 "반올림 오차"만 경계로 클립하고, 실제로 크게 벗어난 box는 계속 오류로 막는다.
            raw_w, raw_h = bw * width, bh * height
            raw_x1 = cx * width - raw_w / 2.0
            raw_y1 = cy * height - raw_h / 2.0
            raw_x2 = raw_x1 + raw_w
            raw_y2 = raw_y1 + raw_h

            # 정규화 좌표 1e-5 수준의 저장/반올림 오차까지만 허용한다.
            boundary_tol_px = max(width, height) * 1e-5  # 960px -> 0.0096px

            if raw_w <= 0 or raw_h <= 0:
                raise RuntimeError(
                    f"Non-positive bbox from {label_path}:{line_no}: "
                    f"{(raw_x1, raw_y1, raw_w, raw_h)}"
                )

            if (
                raw_x1 < -boundary_tol_px
                or raw_y1 < -boundary_tol_px
                or raw_x2 > width + boundary_tol_px
                or raw_y2 > height + boundary_tol_px
            ):
                raise RuntimeError(
                    f"BBox exceeds image by more than rounding tolerance "
                    f"({boundary_tol_px:.6f}px) at {label_path}:{line_no}: "
                    f"{(raw_x1, raw_y1, raw_w, raw_h)}"
                )

            # 반올림 오차 범위에 있는 좌표만 실제 이미지 경계로 클립한다.
            x1 = min(max(raw_x1, 0.0), float(width))
            y1 = min(max(raw_y1, 0.0), float(height))
            x2 = min(max(raw_x2, 0.0), float(width))
            y2 = min(max(raw_y2, 0.0), float(height))
            abs_w = x2 - x1
            abs_h = y2 - y1

            if abs_w <= 0 or abs_h <= 0:
                raise RuntimeError(
                    f"BBox became empty after boundary clipping at "
                    f"{label_path}:{line_no}: {(x1, y1, abs_w, abs_h)}"
                )

            annotations.append({
                "id": ann_id,
                "image_id": image_id,
                "category_id": cls_id,
                "bbox": [float(x1), float(y1), float(abs_w), float(abs_h)],
                "area": float(abs_w * abs_h),
                "iscrowd": 0,
            })
            ann_id += 1
            actual_obj += 1
            object_count += 1

        if actual_obj != expected_obj:
            raise RuntimeError(f"object_count mismatch for {file_name}: manifest={expected_obj}, label={actual_obj}")
        image_id += 1

    return images, annotations, image_id, ann_id, object_count

train_images, train_annotations, next_img, next_ann, train_objects = convert_split(train_files, "train", 1, 1)
val_images, val_annotations, _, _, val_objects = convert_split(val_files, "val", next_img, next_ann)

print('train objects:', train_objects, 'val objects:', val_objects, 'total:', train_objects + val_objects)
if train_objects + val_objects != EXPECTED_OBJECTS:
    raise RuntimeError(f"Total object mismatch: {train_objects + val_objects}")


COCO train:   0%|          | 0/9763 [00:00<?, ?image/s]

COCO val:   0%|          | 0/2433 [00:00<?, ?image/s]

train objects: 37143 val objects: 9251 total: 46394


In [7]:
def make_coco(images, annotations, description):
    return {
        "info": {
            "description": description,
            "pipeline_version": "6.0.5",
            "split_policy": "preserve_exact_upstream",
            "split_fingerprint": SPLIT_FINGERPRINT,
        },
        "licenses": [],
        "images": images,
        "annotations": annotations,
        "categories": categories,
    }

combined = make_coco(train_images + val_images, train_annotations + val_annotations,
                     "YOLO V6.0.5 COCO bridge, fixed upstream Train/Val preserved")
train_coco = make_coco(train_images, train_annotations, "YOLO V6.0.5 Train COCO bridge")
val_coco = make_coco(val_images, val_annotations, "YOLO V6.0.5 Validation COCO bridge")

outputs = {
    LABELS_DIR / "annotations.json": combined,
    LABELS_DIR / "annotations_train.json": train_coco,
    LABELS_DIR / "annotations_val.json": val_coco,
}
for path, obj in outputs.items():
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False)
    print('saved:', path, path.stat().st_size, 'bytes')


saved: /content/drive/MyDrive/baby_kangaroo/week2/데이터전처리/yolo_전처리/pill_yolo_full_v6_0_preprocessed/labels/annotations.json 9294009 bytes
saved: /content/drive/MyDrive/baby_kangaroo/week2/데이터전처리/yolo_전처리/pill_yolo_full_v6_0_preprocessed/labels/annotations_train.json 7435110 bytes
saved: /content/drive/MyDrive/baby_kangaroo/week2/데이터전처리/yolo_전처리/pill_yolo_full_v6_0_preprocessed/labels/annotations_val.json 1868895 bytes


In [8]:
# 최종 계약 검증
assert len(combined["images"]) == EXPECTED_IMAGES
assert len(combined["annotations"]) == EXPECTED_OBJECTS
assert len(combined["categories"]) == EXPECTED_CLASSES
assert len(train_coco["images"]) == EXPECTED_TRAIN
assert len(val_coco["images"]) == EXPECTED_VAL
assert {i["file_name"] for i in train_coco["images"]} == set(train_files)
assert {i["file_name"] for i in val_coco["images"]} == set(val_files)
assert not ({i["file_name"] for i in train_coco["images"]} & {i["file_name"] for i in val_coco["images"]})

print("OK: 12196 images / 46394 objects / 118 classes")
print("OK: fixed Train 9763 / Val 2433 preserved exactly")
print("OK: no new test split created")
print("OK: COCO bridge ready")


OK: 12196 images / 46394 objects / 118 classes
OK: fixed Train 9763 / Val 2433 preserved exactly
OK: no new test split created
OK: COCO bridge ready
